# Exercise 1: Explore Filter Parameters

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import numpy as np
import sys
import os
sys.path.append(os.path.abspath("code"))
from utils import download_data
import lfp_functions as lf

# sns.set_theme(context='notebook',style='white',font_scale=1.5,
#               rc = {'axes.spines.top':False,'axes.spines.right':False,
#                      'image.cmap':plt.cm.jet})

sns.set_theme(context='notebook',style='white',font_scale=1.5,
               rc = {'axes.spines.top':False,'axes.spines.right':False})

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
with open('../data/moving_lfp.pickle', 'rb') as f:
    data = pickle.load(f)

lfp = data['lfp']
sampling_rate = data['sampling_rate']

# take a 5-second excerpt
t_start = 10
t_end = 15
segment = lfp[int(t_start * sampling_rate):int(t_end * sampling_rate)]
t = np.linspace(t_start, t_end, len(segment))

print(f'Sampling rate: {sampling_rate} Hz')
print(f'Segment length: {len(segment)} samples ({t_end - t_start} s)')

Sampling rate: 1000.0 Hz
Segment length: 5000 samples (5 s)


## 1. Effect of bandwidth (fixed center frequency)

I fix the center frequency at 8 Hz (theta band) and vary the bandwidth.

In [ ]:
def bandpass_filter(signal, low_f, high_f, fs, order=5):
    sos = butter(order, [low_f, high_f], btype='band', output='sos', fs=fs)
    return sosfilt(sos, signal)

center = 8  # Hz
half_widths = [0.5, 2, 6]  # narrow -> wide

fig, axes = plt.subplots(len(half_widths) + 1, 1, figsize=(12, 9), sharex=True)

axes[0].plot(t, segment, color='gray', lw=0.8)
axes[0].set_ylabel('raw')
axes[0].set_title('Raw signal and bandpass filters around 8 Hz')

for ax, hw in zip(axes[1:], half_widths):
    low = center - hw
    high = center + hw
    filtered = bandpass_filter(segment, low, high, sampling_rate)
    ax.plot(t, filtered, lw=0.9)
    ax.set_ylabel(f'{low}–{high} Hz')

axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()

**Observation:** narrower bands let through less signal energy and produce a cleaner oscillation, but also attenuate amplitude. Wider bands include more frequencies, making the filtered trace look closer to the raw signal and introducing contributions from neighboring bands.

## 2. Effect of center frequency (fixed bandwidth)

We keep the bandwidth fixed at ±4 Hz and shift the center across the classical LFP bands.

In [ ]:
half_width = 4
centers = [4, 8, 30, 80, 120]  # delta edge, theta, gamma, ripple-adjacent
labels  = ['~4 Hz (delta)', '~8 Hz (theta)', '~30 Hz (slow gamma)',
           '~80 Hz (mid gamma)', '~120 Hz (fast gamma)']

fig, axes = plt.subplots(len(centers) + 1, 1, figsize=(12, 11), sharex=True)

axes[0].plot(t, segment, color='gray', lw=0.8)
axes[0].set_ylabel('raw')
axes[0].set_title('Raw signal and filters at different center frequencies (±4 Hz bandwidth)')

for ax, c, lbl in zip(axes[1:], centers, labels):
    filtered = bandpass_filter(segment, c - half_width, c + half_width, sampling_rate)
    ax.plot(t, filtered, lw=0.9)
    ax.set_ylabel(lbl, fontsize=10)

axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()

**Observation:** the theta band (8 Hz) shows the strongest oscillation — consistent with theta being dominant during locomotion. Higher frequency bands have lower amplitude and faster oscillations. The delta-edge band still captures some slow drift.

## 3. Effect of filter order (fixed band: 6–10 Hz)

Higher-order Butterworth filters have a sharper roll-off but are more expensive to compute.

In [ ]:
low_f, high_f = 6, 10
orders = [1, 3, 5, 10, 20]

fig, axes = plt.subplots(len(orders) + 1, 1, figsize=(12, 11), sharex=True)

axes[0].plot(t, segment, color='gray', lw=0.8)
axes[0].set_ylabel('raw')
axes[0].set_title(f'Filter order comparison — {low_f}–{high_f} Hz band')

for ax, order in zip(axes[1:], orders):
    filtered = bandpass_filter(segment, low_f, high_f, sampling_rate, order=order)
    ax.plot(t, filtered, lw=0.9)
    ax.set_ylabel(f'order {order}')

axes[-1].set_xlabel('time (s)')
plt.tight_layout()
plt.show()

### Computing time vs. filter order

In [ ]:
for order in orders:
    print(f'order {order:2d}:', end=' ')
    %timeit -n 20 -r 3 bandpass_filter(segment, low_f, high_f, sampling_rate, order=order)

**Observation:** increasing the order sharpens the filter's frequency response (the transition from pass-band to stop-band becomes steeper), so leakage from neighboring frequencies is reduced. However, very high orders can introduce ringing artifacts at the edges, and computation time grows roughly linearly with order. For most LFP analyses, orders in the range 4–8 offer a good trade-off.